# 🧹 BÀI TOÁN DỰ ĐOÁN DOANH SỐ GAME 3 NĂM TỚI - BƯỚC 1: LÀM SẠCH & TIỀN XỬ LÝ DỮ LIỆU (`vgsales.csv`)

## 📌 Mục Tiêu Notebook:
1. **Kiểm tra & Xử lý dữ liệu khuyết thiếu (Missing Values)** ở cột `Year` và `Publisher`.
2. **Lọc dữ liệu rác/bị hụt (Outliers & Truncated Data)**: Loại bỏ các năm sau 2016 do dữ liệu cào thiếu hụt.
3. **Phân tích Khám phá Dữ liệu (EDA)**:
   - Đánh giá xu hướng các **Thể loại game (Genre)** tiềm năng.
   - Thống kê **Top 10 tựa game** và nhà phát hành có doanh số ấn tượng.
   - Phân tích tương quan giữa các thị trường: Bắc Mỹ (NA), Châu Âu (EU), Nhật Bản (JP), Khác (Other).
4. **Tạo Feature Chuỗi Thời Gian (Time-Series Features)** & Xuất file dữ liệu sạch `vgsales_cleaned.csv` phục vụ cho mô hình **Machine Learning (XGBoost)** và **Deep Learning (LSTM)**.


## 1. Import Thư Viện & Tải Dữ Liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Cấu hình hiển thị đồ họa
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Tải dữ liệu vgsales.csv
df_raw = pd.read_csv('vgsales.csv')
print(f"Thành công tải dữ liệu! Kích thước ban đầu: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột.")
df_raw.head()

## 2. Kiểm Tra Tổng Quan Dữ Liệu (Data Inspection)

In [ ]:
print("=== Thông tin kiểu dữ liệu và giá trị thiếu ===")
df_raw.info()

print("\n=== Số lượng giá trị Null từng cột ===")
null_counts = df_raw.isnull().sum()
print(null_counts[null_counts > 0])

## 3. Tiền Xử Lý Dữ Liệu & Xử Lý Giá Trị Thiếu (Data Cleaning)
- Cột `Year`: Có 271 dòng thiếu. Ta thử trích xuất số năm từ tên tựa game (ví dụ: 'FIFA 07' -> 2006/2007). Các trường hợp còn lại loại bỏ để bảo vệ tính chính xác của Time-Series.
- Cột `Publisher`: Điền 'Unknown' cho 58 dòng bị khuyết thiếu.

In [ ]:
df = df_raw.copy()

# Function trích xuất năm từ tên tựa game nếu Year bị NaN
def extract_year_from_name(row):
    if pd.isna(row['Year']):
        match = re.search(r'\b(19\d{2}|20\d{2})\b', str(row['Name']))
        if match:
            return float(match.group(1))
        match_short = re.search(r'\b(\d{2})\b', str(row['Name']))
        if match_short:
            yr = int(match_short.group(1))
            if 80 <= yr <= 99:
                return float(1900 + yr)
            elif 0 <= yr <= 20:
                return float(2000 + yr)
    return row['Year']

df['Year'] = df.apply(extract_year_from_name, axis=1)

# Loại bỏ các dòng vẫn bị NaN ở Year
initial_rows = len(df)
df = df.dropna(subset=['Year']).copy()
df['Year'] = df['Year'].astype(int)

# Điền Publisher bị thiếu bằng 'Unknown'
df['Publisher'] = df['Publisher'].fillna('Unknown')

print(f"Đã xử lý xong Null! Số dòng sau khi lọc Year: {len(df)} (Loại bỏ {initial_rows - len(df)} dòng).")

## 4. Lọc Dữ Liệu Thời Gian Tin Cậy (1980 - 2016)
- Dữ liệu từ 2017 - 2020 trong bộ `vgsales.csv` bị sụt giảm bất thường do ngưng thu thập. Ta sẽ lọc giai đoạn **1980 - 2016** để mô hình Time-Series không bị nhiễu.

In [ ]:
print("Doanh số tổng theo năm trước khi lọc:")
print(df.groupby('Year')['Global_Sales'].sum().tail(8))

# Lọc năm <= 2016
df_clean = df[(df['Year'] >= 1980) & (df['Year'] <= 2016)].copy()
print(f"\nKích thước tập dữ liệu sạch (1980-2016): {len(df_clean)} dòng.")

## 5. Phân Tích Khám Phá Dữ Liệu (EDA) & Đánh Giá Tiềm Năng Thể Loại / Top Game

In [ ]:
# 1. Xu hướng doanh số tổng theo năm
yearly_sales = df_clean.groupby('Year')['Global_Sales'].sum().reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=yearly_sales, x='Year', y='Global_Sales', marker='o', color='#2b5c8f', linewidth=2.5)
plt.title('Xu Hướng Tổng Doanh Số Game Toàn Cầu (1980 - 2016)', fontsize=14, fontweight='bold')
plt.xlabel('Năm', fontsize=12)
plt.ylabel('Tổng Doanh Số (Triệu bản/USD)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# 2. Thống kê Thể loại Game (Genre) có tổng doanh số cao nhất
genre_sales = df_clean.groupby('Genre')['Global_Sales'].agg(['sum', 'mean', 'count']).reset_index()
genre_sales = genre_sales.sort_values(by='sum', ascending=False)

plt.figure(figsize=(12, 5))
ax = sns.barplot(data=genre_sales, x='sum', y='Genre', palette='Blues_r')
plt.title('Tổng Doanh Số Theo Thể Loại Game (1980 - 2016)', fontsize=14, fontweight='bold')
plt.xlabel('Tổng Doanh Số toàn cầu (Triệu bản)', fontsize=12)
plt.ylabel('Thể Loại (Genre)', fontsize=12)

for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{width:.1f}M', (width + 10, p.get_y() + p.get_height() / 2.),
                ha='left', va='center', fontsize=10)

plt.xlim(0, genre_sales['sum'].max() * 1.15)
plt.show()

In [ ]:
# 3. Top 10 Tựa Game Có Doanh Số Cao Nhất Lịch Sử
top10_games = df_clean.sort_values(by='Global_Sales', ascending=False).head(10)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=top10_games, x='Global_Sales', y='Name', hue='Genre', dodge=False, palette='viridis')
plt.title('Top 10 Tựa Game Có Doanh Số Cao Nhất (1980 - 2016)', fontsize=14, fontweight='bold')
plt.xlabel('Doanh Số Toàn Cầu (Triệu bản)', fontsize=12)
plt.ylabel('Tên Game', fontsize=12)
plt.legend(title='Thể loại', loc='lower right')
plt.show()

display(top10_games[['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'Global_Sales']])

In [ ]:
# 4. Phân Tích Thể Loại Tiềm Năng Trong Giai Đoạn 2012 - 2016 (Làm cơ sở dự đoán 3 năm tới)
recent_df = df_clean[df_clean['Year'] >= 2012]
recent_genre = recent_df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(12, 5))
sns.barplot(data=recent_genre, x='Global_Sales', y='Genre', palette='magma')
plt.title('Thể Loại Game Có Doanh Thu Cao Nhất Giai Đoạn 2012 - 2016', fontsize=13, fontweight='bold')
plt.xlabel('Tổng Doanh Số (Triệu bản)', fontsize=12)
plt.ylabel('Thể Loại', fontsize=12)
plt.show()

In [ ]:
# 5. Tỷ Trọng Thị Trường (NA vs EU vs JP vs Other)
region_totals = [
    df_clean['NA_Sales'].sum(),
    df_clean['EU_Sales'].sum(),
    df_clean['JP_Sales'].sum(),
    df_clean['Other_Sales'].sum()
]
region_labels = ['Bắc Mỹ (NA)', 'Châu Âu (EU)', 'Nhật Bản (JP)', 'Khác (Other)']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B0']

plt.figure(figsize=(8, 8))
plt.pie(region_totals, labels=region_labels, autopct='%1.1f%%', startangle=140, colors=colors, explode=(0.05, 0, 0, 0))
plt.title('Tỷ Trọng Doanh Số Theo Thị Trường Khu Vực', fontsize=14, fontweight='bold')
plt.show()

## 6. Xuất File Dữ Liệu Đã Làm Sạch (`vgsales_cleaned.csv`)
- Tập dữ liệu này đã được loại bỏ giá trị khuyết thiếu và dữ liệu nhiễu, sẵn sàng làm Input cho bước **Dự đoán doanh số 3 năm tiếp theo** với ML & DL.

In [ ]:
output_path = 'vgsales_cleaned.csv'
df_clean.to_csv(output_path, index=False, encoding='utf-8')
print(f"🎉 Đã xuất thành công file dữ liệu sạch: '{output_path}'!")
print(f"Số bản ghi ghi nhận: {len(df_clean)} dòng.")